# Unsloth Fine-Tuning — ConfessIt

Fine-tune Qwen 2.5 7B on NUS confession-style posts using QLoRA.

- **Base model:** Qwen 2.5 7B (4-bit quantized)
- **Format:** Causal LM (no chat templates) — pure text continuation
- **Data:** ~10k high-engagement posts filtered from SQLite
- **Framework:** Unsloth (2x faster, 60% less VRAM)
- **Runtime:** ~40 min on free Colab T4
- **Export:** Push adapter to HuggingFace Hub (~16 MB)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install git+https://github.com/unslothai/unsloth.git

# Core deps
!pip install transformers datasets accelerate peft trl bitsandbytes

## 2. Upload Dataset

Upload your `confessions.jsonl` to Colab.

In [ ]:
from google.colab import files
import json

print("Upload your confessions.jsonl file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"Loaded: {filename} ({len(uploaded[filename].splitlines())} lines)")

with open("confessions.jsonl", "wb") as f:
    f.write(uploaded[filename])

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="confessions.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}")

texts = [r["text"] for r in train_dataset]
lengths = [len(t) for t in texts]
lengths.sort()
print(f"Text length -> min: {lengths[0]}  |  max: {lengths[-1]}  |  median: {lengths[len(lengths)//2]}")

## 4. Load Base Model (4-bit QLoRA)

In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-7B-bnb-4bit"
max_seq_length = 512  # confessions are short; 512 is plenty

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print(f"Model loaded: {model_name}")
print(f"VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB total — {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free")

## 5. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters injected — trainable params:")
model.print_trainable_parameters()

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    output_dir="outputs",
    report_to="none",
    num_train_epochs=3,
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 7. Inference Demo

In [ ]:
FastLanguageModel.for_inference(model)

prompts = [
    "walked into lecture theatre 27 and realised",
    "can we talk about how",
    "hot take: nus",
    "unpopular opinion but",
]

for prompt in prompts:
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"── {prompt} ──")
    print(result)
    print()

## 8. Push to HuggingFace Hub

The adapter is only ~16 MB — push it straight to HF Hub so you can load it from anywhere.

You'll need a [HuggingFace access token](https://huggingface.co/settings/tokens) (write permission).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Replace with your HF username and desired model name
hf_username = "your-username"  # change this
model_name_hf = "confessit-qwen-2.5-7b-lora"
repo_id = f"{hf_username}/{model_name_hf}"

# Save adapter locally first
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Push to Hub
model.push_to_hub(repo_id, tokenizer=tokenizer, private=False)

print(f"\nPushed to: https://huggingface.co/{repo_id}")
print(f"Model card: https://huggingface.co/{repo_id}/tree/main")
print(f"\nAdapter size: ~16 MB — downloads in seconds from anywhere")

### Optional: Push merged model (4 GB)

If you want a standalone model that doesn't need a base model at inference:

In [ ]:
# Merge adapter into base model and push (~5 min, ~4 GB upload)
# Uncomment if you want the standalone version:

# model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")
# 
# from transformers import AutoModelForCausalLM, AutoTokenizer
# merged = AutoModelForCausalLM.from_pretrained("merged_model")
# merged_tokenizer = AutoTokenizer.from_pretrained("merged_model")
# 
# merged_repo_id = f"{hf_username}/confessit-qwen-2.5-7b-merged"
# merged.push_to_hub(merged_repo_id, tokenizer=merged_tokenizer, private=False)
# print(f"Pushed merged model: https://huggingface.co/{merged_repo_id}")

## 9. Using Your Fine-Tuned Model

### From Python (adapter — recommended)

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-bnb-4bit",
    max_seq_length=512,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=[
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
])

# Load your trained adapter from HF Hub
model.load_adapter("your-username/confessit-qwen-2.5-7b-lora")

FastLanguageModel.for_inference(model)
inputs = tokenizer("hot take: nus", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

### From Python (merged model)

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "your-username/confessit-qwen-2.5-7b-merged",
    torch_dtype="auto",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("your-username/confessit-qwen-2.5-7b-merged")
```

### Via llama.cpp / Ollama (merged model)

If you merged the model, convert to GGUF:

```bash
# Install llama.cpp
pip install llama-cpp-python

# Or convert with llama.cpp's convert.py
# Then quantize: ./quantize confessit-qwen-q4_k_m.gguf Q4_K_M
# Then: ollama create confessit -f Modelfile
```